In [4]:
%%capture
%pip install -U accelerate peft trl bitsandbytes transformers
%pip install evaluate sacrebleu rouge-score bert-score

In [5]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch

In [ ]:
from huggingface_hub import login
login(token="")

# Load Model - Configure and Set Up

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
print("Number of GPUs available:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


Number of GPUs available: 0


In [9]:
# Import Libraries
# Core
import os
import pandas as pd
import numpy as np

# Transformers & Tokenization
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from transformers.trainer_utils import set_seed
from transformers import GenerationConfig
# PEFT
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

# Transformer Reinforcement Learning
from trl import SFTTrainer

# Data
from datasets import Dataset, DatasetDict


set_seed(42) # ensure reproducibility

In [ ]:
 # 4-bit config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True, #recently updated
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

model_name = "BioMistral/BioMistral-7B"
data_name = model_name.split("/")[-2]

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side = "left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # safe default for decoder-only

EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN
tokenizer.padding_side = "left"

# Base model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()

# Evaluate before training

In [ ]:
df = pd.read_json("/content/drive/MyDrive/MIT807/data/silver/question_answers/test.json")
df = df[['instruction', 'question', 'answer', 'context', 'complex_cot', 'options', 'long_answer', 'source', 'category', 'type']]
df.head()

In [ ]:
inference_prompt_style =  """
You are a helpful, respectful, honest, and safe diabetes care assistant.Your answers must be evidence-based, socially unbiased, positive, and emotionally considerate, and must not include harmful, racist, unethical, sexist, toxic, dangerous, or illegal content.
Below is an instruction that describes a task, paired with an input that provides further context.
---
Now answer the next question
## Instruction
{instruction} {context}

## Question
{question} {options}

## Response
...
"""

In [ ]:
def formatting_prompts_func(examples):
    inputs = examples["question"]
    outputs = examples["answer"]
    instructions = examples.get("instruction", [None] * len(inputs)) # Get instructions with default None
    options = examples.get("options", [None] * len(inputs)) # Get options with default None
    contexts = examples.get("context", [None] * len(inputs)) # Get contexts with default None
    cots = examples.get("complex_cot", [None] * len(inputs)) # Get cot with default None
    long_answers = examples.get("long_answer", [None] * len(inputs)) # Get cot with default None
    data_type = examples['type']

    texts = []
    for question, response, instruction, context, option, cot, long_answer, data_type in zip(inputs, outputs, instructions, contexts, options, cots, long_answers, data_type):
        if (instruction is None) or (instruction == "") or (instruction == 'nan') or (instruction == "None"):
            instruction = (
                "You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning."
                "Please answer the following medical question."
            )
        if (context == "nan") or (context == "None") or (context is None) or data_type =='Fill-in-blanks' :
            context = ""
        else:
            context = f"\nContext:\n{context}\n"

        if (option is None) or (option == "None") or (option == 'nan'):
            option = ""
            tags = "(your response must be detailed, consice, clear and well structured diabetes relevant)"
        else:
            tags = "(your response must always follow this format: [OptionLetter]: [OptionText], then start a new line to provide any additional explanation or other details (if applicable).)"
            option = f"\nOptions: {option}"

        text = inference_prompt_style.format(
            instruction=instruction,
            context=context or "",  # Use empty string if context is None
            question=question,
            tags=tags or "",
            explanation=long_answer,
            options=option, # Use empty string if options is None
        )
        texts.append(text + EOS_TOKEN)

    return {"text": texts}

In [ ]:
from datasets import Dataset

# Convert lists in 'Context' column to strings
df['context'] = df['context'].apply(lambda x: str(x) if isinstance(x, list) else x)

# Convert dictionaries in 'options' column to strings
df['options'] = df['options'].apply(lambda x: str(x) if isinstance(x, dict) else x)

dataset = Dataset.from_pandas(df)
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)
print(dataset["text"][1])

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = False

In [ ]:
#GPT Model
from tqdm import tqdm
import csv
import torch
import math
tokenizer.pad_token = tokenizer.eos_token

# configuration
BATCH_SIZE = 50  # tune: try 8,16,32 depending on memory
CSV_PATH = f"/content/drive/MyDrive/MIT807/data/silver/question_answers/output_new/{data_name}_response_test.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
# if using GPU and model supports fp16:
# use_fp16 = (device == "cuda") # This line is no longer needed as .half() is removed

# assume model, tokenizer already loaded
tokenizer.padding_side = "left"
model.to(device)
model.eval()

# if use_fp16: # This block is no longer needed as .half() is removed
#     model.half()

def batch_iterable(iterable, batch_size):
    it = iter(iterable)
    while True:
        batch = []
        try:
            for _ in range(batch_size):
                batch.append(next(it))
        except StopIteration:
            if batch:
                yield batch
            break
        yield batch

# Write header once
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["input", "output"])

# Process in batches and append to CSV
total = len(dataset["text"])
with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    for batch in tqdm(batch_iterable(dataset["text"], BATCH_SIZE), desc="Generating (batched)", total=math.ceil(total/BATCH_SIZE)):
        # batch tokenization (adjust to your tokenizer API)
        encoding = tokenizer(batch,
                             return_tensors="pt",
                             padding=True,
                             truncation=True,
                             max_length=model.config.max_position_embeddings)

        # --- SAFETY CHECK: token id range ---
        if encoding["input_ids"].max().item() >= model.config.vocab_size:
            raise RuntimeError(
                f"Token id {int(encoding['input_ids'].max())} >= model.vocab_size ({model.config.vocab_size}). "
                "Fix by loading the tokenizer matching the model checkpoint."
            )

        # Move tensors to device
        inputs = {k: v.to(device) if torch.is_tensor(v) else v for k, v in encoding.items()}

        # Calculate max_new_tokens ensuring it is at least 1
        max_new = max(1, model.config.max_position_embeddings - inputs["input_ids"].shape[1])


        # faster inference context
        with torch.inference_mode():
            # If your generate_response wraps a custom generation, adapt this:
            outputs = model.generate(**inputs,
                                     max_new_tokens=max_new,
                                     use_cache=True,
                                     do_sample=True,     # enable stochastic sampling
                                     temperature=0.7,    # reduce overfitting to repeated patterns
                                     top_p=0.9,
                                     top_k=50,
                                     repetition_penalty=1.2,
                                     pad_token_id=tokenizer.eos_token_id)  # tweak
            decoded = tokenizer.batch_decode(outputs,
                                             skip_special_tokens=True)
        torch.cuda.empty_cache()
        # stream to disk
        for inp, out in zip(batch, decoded):
            writer.writerow([inp, out])

# Load Data Training

In [ ]:
train_prompt_style = """
<system>
You are a helpful, respectful, honest, and safe diabetes care assistant.Your answers must be evidence-based, socially unbiased, positive, and emotionally considerate, and must not include harmful, racist, unethical, sexist, toxic, dangerous, or illegal content.
Below is an instruction that describes a task, paired with an input that provides further context.
</system>
<user>
# Instruction
{instruction}
{context}
# Question
{question} {options}
</user>
# Answer <assistant>
{response} {explanation} <assistant>
"""

In [ ]:
def formatting_prompts_func(examples):
    inputs = examples["question"]
    outputs = examples["answer"]
    instructions = examples.get("instruction", [None] * len(inputs)) # Get instructions with default None
    options = examples.get("options", [None] * len(inputs)) # Get options with default None
    contexts = examples.get("context", [None] * len(inputs)) # Get contexts with default None
    long_answers = examples.get("long_answer", [None] * len(inputs)) # Get cot with default None
    data_type = examples['type']

    texts = []
    for question, response, instruction, context, option, long_answer, data_type in zip(inputs, outputs, instructions, contexts, options, long_answers, data_type):
        if (instruction is None) or (instruction == "") or (instruction == 'nan') or (instruction == "None"):
            instruction = (
                "You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning."
                "Please answer the following medical question."
            )
        if (context == "nan") or (context == "None") or (context is None) or data_type =='Fill-in-blanks' :
            context = ""
        else:
            context = f"\n# Context:\n{context}\n"

        if (long_answer is None) or (long_answer == "None") or (long_answer == "nan"):
            long_answer = ""
        else:
            long_answer = f"\n{long_answer}"

        if (option is None) or (option == "None") or (option == 'nan'):
            option = ""
        else:
            option = f"\nOptions: {option}"

        text = train_prompt_style.format(
            instruction=instruction,
            context=context or "",  # Use empty string if context is None
            question=question,
            explanation=long_answer,
            options=option, # Use empty string if options is None
            response=response,
        )
        texts.append(text + EOS_TOKEN)

    return {"text": texts}

In [ ]:
df = pd.read_json("/content/drive/MyDrive/MIT807/data/silver/question_answers/training.json")
df = df[['instruction', 'question', 'answer', 'context', 'complex_cot', 'category', 'options', 'long_answer', 'type']]
df.info()
df.head(1)

In [ ]:
# Convert lists in 'Context' column to strings
df['context'] = df['context'].apply(lambda x: str(x) if isinstance(x, list) else x)

# Convert dictionaries in 'options' column to strings
df['options'] = df['options'].apply(lambda x: str(x) if isinstance(x, dict) else x)

dataset = Dataset.from_pandas(df)
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)
print(dataset["text"][1])

In [ ]:
print(dataset["text"][10000])

In [ ]:
#split to test, evaluation and training
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset["validation"] = dataset["test"]
del dataset["test"]
#dataset["train"][0] # Removed this line
train_dataset = dataset["train"] # Added this line
eval_dataset = dataset["validation"] # Added this line

In [ ]:
dataset

In [ ]:
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
print(eval_dataset['text'][5])

# PEFT Implementation

In [ ]:
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
from peft import (
    PromptTuningConfig,
    TaskType,
    get_peft_model,
    PromptTuningInit,
    prepare_model_for_kbit_training # Corrected import
)
# First, apply prompt tuning
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=20,
    tokenizer_name_or_path=model_name,
)

# Now prepare for k-bit training
model = prepare_model_for_kbit_training( # Corrected function call
    model,
    use_gradient_checkpointing=True, # Keep this enabled as it's needed for prepare_model_for_kbit_training
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
# The following line is commented out as the model is already loaded with quantization config
model = get_peft_model(model, peft_config)

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable} / {total} ({100*trainable/total:.4f}%)")

In [ ]:
from transformers import TrainerCallback
class SaveAtEpochEndCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        # this flag tells Trainer to save at the end of each epoch
        control.should_save = True
        return control

In [ ]:
output_dir = f"/content/drive/MyDrive/MIT807/new_{data_name}_prompt_tuning/"
os.makedirs(output_dir, exist_ok=True)
# Training Arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=8,   # reduce if OOM; use grad_accumulation to enlarge effective batch size
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    max_steps=500,                    # use epochs-based termination; set if you prefer step-based
    learning_rate=1.6e-4,              # safer starting point; reduce if unstable
    lr_scheduler_type="cosine",      #"linear", #"constant", #recently added
    weight_decay=1.6e-5,
    optim="paged_adamw_32bit",        # requires bitsandbytes/paged optimizer in your env
    logging_strategy="steps",
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    bf16=(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    fp16=(not torch.cuda.is_bf16_supported() and torch.cuda.is_available()),
    gradient_checkpointing=True,
    report_to=None,
)

# Train

In [ ]:
from peft import prepare_model_for_kbit_training, get_peft_model
# Initialize the Trainer
trainer = SFTTrainer(
    model,
    args=training_args,
    train_dataset=train_dataset, # Corrected line
    data_collator=data_collator,
    eval_dataset=eval_dataset, # Added this line
    callbacks=[
        SaveAtEpochEndCallback
    ]
)

In [38]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable} / {total} ({100*trainable/total:.4f}%)")

Trainable params: 0 / 1335956480 (0.0000%)


In [ ]:
# train
import gc, torch
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = False
trainer.train()#resume_from_checkpoint=True)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=500, training_loss=1.0312340850830077, metrics={'train_runtime': 6694.5925, 'train_samples_per_second': 1.494, 'train_steps_per_second': 0.075, 'total_flos': 3.832637717282611e+17, 'train_loss': 1.0312340850830077, 'epoch': 0.7757951900698216})

In [ ]:
# Save new model
new_model = "/content/drive/MyDrive/MIT807/biomedlm_qlora_fine_tuned/"
trainer.model.save_pretrained(new_model)
data_name = new_model.split("/")[-2]
data_name

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = False

# Evaluate performance

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

In [ ]:
fine_model = PeftModel.from_pretrained(model, new_model)
# Move the model to bfloat16 after loading
fine_model.to(torch.bfloat16)
# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
print(dataset["text"][2])

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = False

In [ ]:
from tqdm import tqdm
import csv
import math
import os

BATCH_SIZE = 50#20
CSV_PATH = f"/content/drive/MyDrive/MIT807/data/silver/question_answers/output_new/{data_name}_response_test_v2.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer.padding_side = "left"
fine_model.to(device) # Ensure fine_model is on the correct device
fine_model.eval()

# Resume processing
processed = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        # Check if the file is not empty before trying to read the header
        if os.path.getsize(CSV_PATH) > 0:
            try:
                next(reader) # Skip header row if exists
            except StopIteration:
                pass  # Handle empty file or file with only header


            for row in reader:
                # Ensure row has enough elements before accessing row[0]
                if row and len(row) > 0:
                    processed.add(row[0])


dataset_to_process = [x for x in dataset["text"] if x not in processed]

def batch_iterable(iterable, batch_size):
    it = iter(iterable)
    while True:
        batch = []
        try:
            for _ in range(batch_size):
                batch.append(next(it))
        except StopIteration:
            if batch:
                yield batch
            break
        yield batch

with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    total = len(dataset_to_process)
    for batch in tqdm(batch_iterable(dataset_to_process, BATCH_SIZE), desc="Generating (batched)", total=math.ceil(total/BATCH_SIZE)):
        # batch tokenization (adjust to your tokenizer API)
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(device)

        # No explicit casting needed here if fine_model is already in bfloat16

        # faster inference context
        with torch.inference_mode():
            # If your generate_response wraps a custom generation, adapt this:
            outputs = fine_model.generate(
                input_ids=inputs["input_ids"],
                max_new_tokens=256,
                attention_mask=inputs["attention_mask"],
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
                repetition_penalty=1.2,
                do_sample=False
                )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=False)
        torch.cuda.empty_cache()
        # stream to disk
        for inp, out in zip(batch, decoded):
            writer.writerow([inp, out])

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = False

In [ ]:
# idle for 5 second
import time
time.sleep(5)